# Prot2Text-V2 + Foldseek retrieval: training and generation

Pipeline: download AlphaFold structures -> Foldseek retrieval -> retrieval-augmented prompts -> LoRA fine-tune -> generate (with / without retrieval / sequence-only) -> metrics.

All logic lives in `src/prot2text_foldseek/`; this notebook only configures and calls it. Every tunable value is in `config.py` (edit `cfg` below to override).

**Setup**: put the `src/` folder in `<drive_root>/src` and `requirements.txt` next to it.

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
!pip install -q -r /content/drive/MyDrive/Colab_Notebooks/266_Final_Project/src/requirements.txt --upgrade

In [ ]:
# Foldseek binary (not on PyPI)
!wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
!tar xzf foldseek-linux-avx2.tar.gz

import os
os.environ["PATH"] += ":/content/foldseek/bin"

In [ ]:
import logging
import os
import random
import sys

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, "/content/drive/MyDrive/Colab_Notebooks/266_Final_Project/src")   # make the package importable

from prot2text_foldseek import dataset, generation, metrics, modeling, retrieval, structures, training
from prot2text_foldseek.config import Config

logging.basicConfig(level=logging.INFO, format="%(message)s")

cfg = Config()          # edit fields here, e.g. cfg.train.num_epochs = 2
paths = cfg.paths

os.makedirs(paths.scratch_dir, exist_ok=True)
os.makedirs(paths.out_dir, exist_ok=True)
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

print("scratch:", paths.scratch_dir, "\nout    :", paths.out_dir)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu: {props.name}  {props.total_memory / 1e9:.1f} GB")

## Load data

CSV columns used: `AlphaFoldDB, Full Name, taxon, sequence, function`.

In [ ]:
raw = {s: pd.read_csv(os.path.join(paths.csv_dir, f"{s}.csv")) for s in cfg.splits}
for s, df in raw.items():
    print(f"{s:5s} n={len(df):>7,}  cols={list(df.columns)}")

## Part 1 - Structures and Foldseek retrieval

If the index and hit tables are already on Drive they are restored; otherwise structures are downloaded, the index is built, every split is searched against the train index, and the results are backed up to Drive.

In [ ]:
if retrieval.backup_exists(paths.index_backup_dir, cfg.splits):
    index_db, m8 = retrieval.restore_artifacts(paths.index_backup_dir, paths.scratch_dir, cfg.splits)
else:
    pdb_dirs, failed = structures.download_all(
        raw, cfg.splits, paths.pdb_root, cfg.download, failed_path=paths.failed_downloads_path)

    index_db = paths.index_db
    retrieval.build_index(pdb_dirs["train"], index_db, cfg.foldseek)

    # Search every split against the train index:
    #   train     -> nearest training neighbours
    #   eval/test -> similar training proteins whose descriptions populate the prompts
    m8 = {
        s: retrieval.search_split(
            s, pdb_dirs[s], index_db, paths.hits_m8(s),
            os.path.join(paths.scratch_dir, f"tmp_{s}"), cfg.foldseek)
        for s in cfg.splits
    }
    retrieval.backup_artifacts(index_db, m8, paths.index_backup_dir)

### Parse hits and downsample easy training hits

In [ ]:
train_desc = retrieval.build_train_desc(raw["train"])
hit_taxon_lookup = retrieval.build_taxon_lookup(raw["train"])
print(f"{len(train_desc):,} training descriptions | {len(hit_taxon_lookup):,} taxa")

hits = {
    s: retrieval.parse_hits(m8[s], train_desc, cfg.foldseek.columns, cfg.retrieval.keep_top_hits)
    for s in cfg.splits
}
retrieval.assert_no_self_hits(hits)

# Eval/test hit rates are expected to be lower than train
for s in cfg.splits:
    print(retrieval.summarize_hits(hits[s], len(raw[s]), s))

In [ ]:
# Easy train hits (identity > threshold) are mostly dropped so the model can't just copy
prompt_hits = retrieval.build_prompt_hits(hits, cfg.splits, cfg.retrieval, cfg.seed)
retrieval.save_prompt_hits(prompt_hits, paths.prompt_hits_path)

for s in cfg.splits:
    best = np.array([h[0]["fident"] for h in prompt_hits[s].values() if h])
    frac = len(best) / max(len(raw[s]), 1)
    p25, p50, p75 = np.percentile(best, [25, 50, 75])
    print(f"{s:5s} after downsampling: {frac:5.1%} have >=1 hit; fident p25/50/75 = {p25:.2f} / {p50:.2f} / {p75:.2f}")

## Datasets and collaters

The collater builds the Llama chat prompt (`Taxon ; Similar structures ; Sequence embeddings`) and tokenises the ESM2 input.

In [ ]:
esm_tokenizer, llama_tokenizer = dataset.load_tokenizers(cfg.models.esm_path, cfg.models.llama_path)

factory = dataset.SplitFactory(
    raw, prompt_hits, esm_tokenizer, llama_tokenizer, hit_taxon_lookup,
    cfg.data, cfg.retrieval, cfg.seed)

train_ds, train_col = factory.make("train", "train", trim=cfg.data.trim_train)
# Eval loss should be deterministic: no taxonomy/retrieval dropout and no random cropping
eval_ds, eval_col = factory.make("eval", "train", taxonomy_dropout=0.0, retrieval_dropout=0.0,
                                 random_crop=False, trim=cfg.data.trim_eval)
print(f"train {len(train_ds):,} | eval {len(eval_ds):,}")

In [ ]:
# Sanity check: read an actual prompt before spending GPU hours on it
probe_ds, probe_col = factory.make("test", "inference", taxonomy_dropout=0.0,
                                   retrieval_dropout=0.0, trim=8)
probe_batch = probe_col([probe_ds[i] for i in range(4)])
text, shapes = dataset.inspect_batch(probe_batch, llama_tokenizer)   # also asserts placeholder counts
print(text)
print("\n--- shapes ---")
for name, shape in shapes.items():
    print(f"{name:28s} {shape}")

## Model

In [ ]:
device = "cuda"
model = modeling.build_model(cfg.models.hf_model_id, paths.repo_root, cfg.lora, device=device)

## Training

In [ ]:
train_loader, eval_loader = training.build_dataloaders(train_ds, train_col, eval_ds, eval_col, cfg.train)

In [ ]:
history = training.fit(model, train_loader, eval_loader, cfg.train, paths.out_dir, device=device)
history

## Prediction generation

Three conditions on the test split: full retrieval prompt, taxonomy only (no retrieval), and sequence only (no taxonomy, no retrieval).

In [ ]:
gen_model = generation.merge_for_inference(model)

gens = {
    name: generation.generate_split(
        gen_model, factory, llama_tokenizer, "test", condition,
        cfg.generation, paths.out_dir, trim=cfg.data.trim_test, device=device)
    for name, condition in generation.CONDITIONS.items()
}

for acc in list(gens["with"])[:2]:
    print("=" * 70)
    print("ACC :", acc)
    print("TRUE:", gens["with"][acc]["true"][:400])
    print("PRED:", gens["with"][acc]["pred"][:400])

## Evaluation

In [ ]:
import json

all_metrics = {}
for name in ("with", "without", "seq_only"):
    print(f"\n### {name.upper()} ###")
    _, all_metrics[name] = metrics.score_results(gens[name])

# Same file layout the analysis notebook reads
with open(os.path.join(paths.out_dir, "metrics.json"), "w") as fh:
    json.dump({"with_retrieval": all_metrics["with"], "without_retrieval": all_metrics["without"]}, fh, indent=2)
with open(os.path.join(paths.out_dir, "metrics_seq_only.json"), "w") as fh:
    json.dump(all_metrics["seq_only"], fh, indent=2)

table = metrics.comparison_table(all_metrics)
table.loc["delta (with - without)"] = table.loc["with"] - table.loc["without"]
table

### Stratified by best-hit identity (with vs. without retrieval)

In [ ]:
references = {a: v["true"] for a, v in gens["with"].items()}
predictions = {k: {a: v["pred"] for a, v in g.items()} for k, g in gens.items()}
bin_of = {a: metrics.identity_bin(metrics.best_fident(prompt_hits["test"], a)) for a in references}

strat = metrics.stratified_metrics(references, predictions, bin_of, treatment="with", control="without")
strat.to_csv(os.path.join(paths.out_dir, "stratified_metrics.csv"), index=False)
strat